# Preprocessing

In [10]:
import os
import pandas as pd
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def load_ecg_data(data_dir):
    """
    Load ECG data from the specified directory.

    Parameters:
    data_dir (str): Path to the directory containing ECG data files.

    Returns:
    list: A list of tuples, each containing the ECG signal and its corresponding label.
    """
    # Load the labels from REFERENCE.csv
    reference_path = os.path.join(data_dir, 'REFERENCE.csv')
    labels_df = pd.read_csv(reference_path, header=None, names=['record', 'label'])
    labels_dict = dict(zip(labels_df['record'], labels_df['label']))

    ecg_data = []
    for file_name in os.listdir(data_dir):
        if file_name.endswith('.mat'):
            record_name = file_name.replace('.mat', '')
            
            if record_name not in labels_dict:
                continue  # security if one .mat does not have an associated label
            
            file_path = os.path.join(data_dir, file_name)
            mat_data = loadmat(file_path)
            signal = mat_data['val'][0]
            label = labels_dict[record_name]
            
            ecg_data.append((signal, label))
    
    return ecg_data

### Data length normalization

In [4]:
def data_normalization(data, target_len=9000, overlap=0.5):
    data = np.array(data)
    segments = []
    
    if len(data) > target_len:
        # Chop recording into 9000 samples with 50% overlap between segments
        step = int(target_len * (1 - overlap))  # 4500
        start = 0
        while start + target_len <= len(data):
            segment = data[start:start + target_len]
            segments.append(segment)
            start += step
    
    elif len(data) < target_len:
        # Append DATA in the back of the recording until reaching 9000 samples
        DATA = data.copy()
        repeated = data.copy()
        while len(repeated) < target_len:
            repeated = np.concatenate([repeated, DATA])
        segment = repeated[:target_len] 
        segments.append(segment)
    
    else:
        # 7-8: IF the length of the recording is equal to 9000 samples -> Preserve
        segments.append(data)
    
    return segments

### Normalizagion of the length

In [ ]:
ecg_data = load_ecg_data('../data/training2017')

all_segments = []
all_labels = []

for signal, label in ecg_data:
    segments = data_normalization(signal)
    all_segments.extend(segments)
    all_labels.extend([label] * len(segments))

print(f"Number of total segments : {len(all_segments)}")

Number of total segments : 10416


### Check the normalization number

In [13]:
print(pd.Series(all_labels).value_counts())

N    5972
O    3211
A     933
~     300
Name: count, dtype: int64


### Save the preprocessed segments

In [15]:
import numpy as np

np.save('../data/processed/segments.npy', np.array(all_segments, dtype=object))
np.save('../data/processed/labels.npy', np.array(all_labels))